In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

sample_7d = pd.read_csv(
    'step2_click_sample_labeled_7d.csv',
    parse_dates=['click_date', 'first_click_ts', 'last_click_ts', 'order_time'],
    low_memory=False
)

print(sample_7d.shape)
print(sample_7d.columns.tolist())
sample_7d.head()

(213050, 41)
['user_id', 'sku_id', 'channel', 'click_date', 'first_click_ts', 'last_click_ts', 'click_cnt', 'hour', 'dow', 'is_weekend', 'hour_bucket', 'user_day_total_clicks', 'user_day_distinct_skus', 'user_day_distinct_channels', 'user_sku_click_days_so_far', 'user_sku_clicks_so_far', 'repeat_same_sku_flag', 'same_day_multi_click_flag', 'same_sku_click_share_in_day', 'sku_focus_flag', 'is_plus', 'purchase_power', 'city_level', 'gender', 'age', 'user_level', 'type', 'brand_id', 'any_attr_missing', 'any_attr_placeholder', 'brand_sku_cnt', 'user_sku_key', 'order_time', 'order_id', 'order_gmv', 'order_qty', 'days_to_purchase', 'label_1d', 'label_3d', 'label_7d', 'click_cnt_bucket']


,user_id,sku_id,channel,click_date,first_click_ts,last_click_ts,click_cnt,hour,dow,is_weekend,hour_bucket,user_day_total_clicks,user_day_distinct_skus,user_day_distinct_channels,user_sku_click_days_so_far,user_sku_clicks_so_far,repeat_same_sku_flag,same_day_multi_click_flag,same_sku_click_share_in_day,sku_focus_flag,is_plus,purchase_power,city_level,gender,age,user_level,type,brand_id,any_attr_missing,any_attr_placeholder,brand_sku_cnt,user_sku_key,order_time,order_id,order_gmv,order_qty,days_to_purchase,label_1d,label_3d,label_7d,click_cnt_bucket
0,7a89b29ba5,a0e49f9966,app,2018-02-28,2018-02-28 23:59:01,2018-02-28 23:59:01,1,23.0,2.0,0,late_night,3,2,1,1,1,0,0,0.333333,0,1,2,1,F,26-35,4,1,7cc01be867,0,1,253,7a89b29ba5||a0e49f9966,2018-03-02 10:08:04,79cded2f9d,-1.0,4.0,1.422951,0,1,1,1
1,ba189a22b7,2f268cf558,app,2018-02-28,2018-02-28 23:59:02,2018-02-28 23:59:02,1,23.0,2.0,0,late_night,1,1,1,1,1,0,0,1.000000,1,0,2,4,F,26-35,2,2,603dc9ab6b,0,0,290,ba189a22b7||2f268cf558,NaT,NaN,NaN,NaN,NaN,0,0,0,1
2,e03f8c6d4e,6a0f1004bb,app,2018-02-28,2018-02-28 23:59:02,2018-02-28 23:59:02,1,23.0,2.0,0,late_night,1,1,1,1,1,0,0,1.000000,1,1,2,4,F,26-35,4,1,8368e30fa1,0,0,5,e03f8c6d4e||6a0f1004bb,NaT,NaN,NaN,NaN,NaN,0,0,0,1
3,7093c5f021,861f71f9ad,app,2018-02-28,2018-02-28 23:59:04,2018-02-28 23:59:04,1,23.0,2.0,0,late_night,1,1,1,1,1,0,0,1.000000,1,0,3,1,M,16-25,1,2,dc7e4ae5fc,0,0,419,7093c5f021||861f71f9ad,NaT,NaN,NaN,NaN,NaN,0,0,0,1
4,7492b76bdf,d3e31fdd6e,app,2018-02-28,2018-02-28 23:59:04,2018-02-28 23:59:04,1,23.0,2.0,0,late_night,5,2,1,1,1,0,0,0.200000,0,0,4,4,M,26-35,3,1,99d41501ff,0,0,271,7492b76bdf||d3e31fdd6e,2018-03-01 00:15:48,d23139c202,0.0,1.0,0.011620,1,1,1,1


In [2]:
analysis_df = sample_7d[
    [
        'label_7d',
        'label_3d',
        'label_1d',
        'channel',
        'type',
        'sku_focus_flag',
        'repeat_same_sku_flag',
        'click_cnt',
        'user_day_distinct_skus',
        'user_day_distinct_channels',
        'is_plus',
        'purchase_power',
        'city_level',
        'gender',
        'age'
    ]
].copy()

analysis_df = analysis_df.dropna(subset=[
    'label_7d',
    'channel',
    'type',
    'sku_focus_flag',
    'click_cnt',
    'user_day_distinct_skus',
    'user_day_distinct_channels'
]).copy()

print(analysis_df.shape)
analysis_df.head()

(213050, 15)


,label_7d,label_3d,label_1d,channel,type,sku_focus_flag,repeat_same_sku_flag,click_cnt,user_day_distinct_skus,user_day_distinct_channels,is_plus,purchase_power,city_level,gender,age
0,1,1,0,app,1,0,0,1,2,1,1,2,1,F,26-35
1,0,0,0,app,2,1,0,1,1,1,0,2,4,F,26-35
2,0,0,0,app,1,1,0,1,1,1,1,2,4,F,26-35
3,0,0,0,app,2,1,0,1,1,1,0,3,1,M,16-25
4,1,1,1,app,1,0,0,1,2,1,0,4,4,M,26-35


In [3]:
target_label = 'label_7d'

print('整体7天转化率:', analysis_df[target_label].mean())

整体7天转化率: 0.13928185871861065


In [4]:
# 4.1 点击次数分桶
analysis_df['click_cnt_bucket'] = pd.cut(
    analysis_df['click_cnt'],
    bins=[0, 1, 2, 3, 5, 10, 999999],
    labels=['1', '2', '3', '4-5', '6-10', '10+'],
    right=True
)

# 4.2 当天浏览SKU数分桶
analysis_df['distinct_sku_bucket'] = pd.cut(
    analysis_df['user_day_distinct_skus'],
    bins=[0, 2, 5, 10, 20, 999999],
    labels=['1-2', '3-5', '6-10', '11-20', '20+'],
    right=True
)

# 4.3 当天跨渠道数分桶
analysis_df['distinct_channel_bucket'] = pd.cut(
    analysis_df['user_day_distinct_channels'],
    bins=[0, 1, 2, 3, 10],
    labels=['1', '2', '3', '4+'],
    right=True
)

# 4.4 更直观的类型标签（可选）
analysis_df['type_name'] = analysis_df['type'].map({1: '1P', 2: '3P'})
analysis_df['intent_group'] = analysis_df[target_label].map({0: '低意愿点击', 1: '高意愿点击'})

analysis_df.head()

,label_7d,label_3d,label_1d,channel,type,sku_focus_flag,repeat_same_sku_flag,click_cnt,user_day_distinct_skus,user_day_distinct_channels,is_plus,purchase_power,city_level,gender,age,click_cnt_bucket,distinct_sku_bucket,distinct_channel_bucket,type_name,intent_group
0,1,1,0,app,1,0,0,1,2,1,1,2,1,F,26-35,1,1-2,1,1P,高意愿点击
1,0,0,0,app,2,1,0,1,1,1,0,2,4,F,26-35,1,1-2,1,3P,低意愿点击
2,0,0,0,app,1,1,0,1,1,1,1,2,4,F,26-35,1,1-2,1,1P,低意愿点击
3,0,0,0,app,2,1,0,1,1,1,0,3,1,M,16-25,1,1-2,1,3P,低意愿点击
4,1,1,1,app,1,0,0,1,2,1,0,4,4,M,26-35,1,1-2,1,1P,高意愿点击


In [5]:
portrait_numeric = (
    analysis_df
    .groupby('intent_group', as_index=False)
    .agg(
        samples=(target_label, 'size'),
        conv_rate=(target_label, 'mean'),
        avg_click_cnt=('click_cnt', 'mean'),
        avg_distinct_skus=('user_day_distinct_skus', 'mean'),
        avg_distinct_channels=('user_day_distinct_channels', 'mean'),
        focus_rate=('sku_focus_flag', 'mean'),
        repeat_rate=('repeat_same_sku_flag', 'mean'),
        plus_rate=('is_plus', 'mean')
    )
)

display(portrait_numeric)

,intent_group,samples,conv_rate,avg_click_cnt,avg_distinct_skus,avg_distinct_channels,focus_rate,repeat_rate,plus_rate
0,低意愿点击,183376,0.0,2.244841,8.894528,1.080981,0.17709,0.074933,0.261599
1,高意愿点击,29674,1.0,3.590685,4.471187,1.092842,0.51075,0.086877,0.268383


In [6]:
portrait_channel = pd.crosstab(
    analysis_df['channel'],
    analysis_df['intent_group'],
    normalize='columns'
).reset_index()

display(portrait_channel)

intent_group,channel,低意愿点击,高意愿点击
0,app,0.840939,0.810440
1,mobile,0.016005,0.028948
2,others,0.011572,0.014086
3,pc,0.047302,0.070803
4,wechat,0.084182,0.075723


In [7]:
portrait_channel_cnt = pd.crosstab(
    analysis_df['channel'],
    analysis_df['intent_group']
).reset_index()

display(portrait_channel_cnt)

intent_group,channel,低意愿点击,高意愿点击
0,app,154208,24049
1,mobile,2935,859
2,others,2122,418
3,pc,8674,2101
4,wechat,15437,2247


In [8]:
portrait_type = pd.crosstab(
    analysis_df['type_name'],
    analysis_df['intent_group'],
    normalize='columns'
).reset_index()

display(portrait_type)

intent_group,type_name,低意愿点击,高意愿点击
0,1P,0.607348,0.726663
1,3P,0.392652,0.273337


In [9]:
portrait_type_cnt = pd.crosstab(
    analysis_df['type_name'],
    analysis_df['intent_group']
).reset_index()

display(portrait_type_cnt)

intent_group,type_name,低意愿点击,高意愿点击
0,1P,111373,21563
1,3P,72003,8111


In [10]:
portrait_purchase_power = pd.crosstab(
    analysis_df['purchase_power'],
    analysis_df['intent_group'],
    normalize='columns'
).reset_index()

display(portrait_purchase_power)

intent_group,purchase_power,低意愿点击,高意愿点击
0,-1,0.148705,0.140022
1,1,0.012166,0.014524
2,2,0.591195,0.608310
3,3,0.229474,0.217497
4,4,0.018154,0.019209
5,5,0.000305,0.000438


In [11]:
portrait_city = pd.crosstab(
    analysis_df['city_level'],
    analysis_df['intent_group'],
    normalize='columns'
).reset_index()

display(portrait_city)

intent_group,city_level,低意愿点击,高意愿点击
0,-1,0.150401,0.137562
1,1,0.220143,0.251230
2,2,0.344456,0.341612
3,3,0.138911,0.136483
4,4,0.133033,0.119903
5,5,0.013055,0.013210


In [12]:
portrait_age = pd.crosstab(
    analysis_df['age'],
    analysis_df['intent_group'],
    normalize='columns'
).reset_index()

display(portrait_age)

intent_group,age,低意愿点击,高意愿点击
0,16-25,0.280424,0.242637
1,26-35,0.411526,0.436443
2,36-45,0.168424,0.185449
3,46-55,0.033129,0.034542
4,<=15,0.000065,0.000000
5,>=56,0.020892,0.022882
6,U,0.085540,0.078048


In [13]:
portrait_gender = pd.crosstab(
    analysis_df['gender'],
    analysis_df['intent_group'],
    normalize='columns'
).reset_index()

display(portrait_gender)

intent_group,gender,低意愿点击,高意愿点击
0,F,0.717967,0.716149
1,M,0.195347,0.204624
2,U,0.086685,0.079228


In [14]:
portrait_diff = (
    analysis_df
    .groupby(target_label)
    .agg(
        avg_click_cnt=('click_cnt', 'mean'),
        avg_distinct_skus=('user_day_distinct_skus', 'mean'),
        avg_distinct_channels=('user_day_distinct_channels', 'mean'),
        focus_rate=('sku_focus_flag', 'mean'),
        repeat_rate=('repeat_same_sku_flag', 'mean'),
        plus_rate=('is_plus', 'mean')
    )
    .T
)

portrait_diff.columns = ['低意愿点击', '高意愿点击']
portrait_diff['差值(高-低)'] = portrait_diff['高意愿点击'] - portrait_diff['低意愿点击']

display(portrait_diff.sort_values('差值(高-低)', ascending=False))

,低意愿点击,高意愿点击,差值(高-低)
avg_click_cnt,2.244841,3.590685,1.345844
focus_rate,0.177090,0.510750,0.333660
repeat_rate,0.074933,0.086877,0.011944
avg_distinct_channels,1.080981,1.092842,0.011861
plus_rate,0.261599,0.268383,0.006784
avg_distinct_skus,8.894528,4.471187,-4.423341


In [15]:
combo_focus_click = (
    analysis_df
    .groupby(['sku_focus_flag', 'click_cnt_bucket'], observed=True, as_index=False)
    .agg(
        samples=(target_label, 'size'),
        conv_7d=(target_label, 'mean')
    )
    .sort_values(['sku_focus_flag', 'click_cnt_bucket'])
)

display(combo_focus_click)

,sku_focus_flag,click_cnt_bucket,samples,conv_7d
0,0,1,96556,0.046771
1,0,2,30963,0.099570
2,0,3,14308,0.146631
3,0,4-5,12645,0.178648
4,0,6-10,8421,0.234770
5,0,10+,2527,0.231500
6,1,1,18753,0.245241
7,1,2,9023,0.330045
8,1,3,5614,0.383862
9,1,4-5,6406,0.385264


In [16]:
combo_focus_sku = (
    analysis_df
    .groupby(['sku_focus_flag', 'distinct_sku_bucket'], observed=True, as_index=False)
    .agg(
        samples=(target_label, 'size'),
        conv_7d=(target_label, 'mean')
    )
    .sort_values(['sku_focus_flag', 'distinct_sku_bucket'])
)

display(combo_focus_sku)

,sku_focus_flag,distinct_sku_bucket,samples,conv_7d
0,0,1-2,8569,0.153460
1,0,3-5,49603,0.120577
2,0,6-10,53448,0.087506
3,0,11-20,37177,0.055195
4,0,20+,16623,0.029658
5,1,1-2,38794,0.309352
6,1,3-5,7930,0.356494
7,1,6-10,854,0.365340
8,1,11-20,51,0.313725
9,1,20+,1,0.000000


In [17]:
combo_focus_click_sku = (
    analysis_df
    .groupby(
        ['sku_focus_flag', 'click_cnt_bucket', 'distinct_sku_bucket'],
        observed=True,
        as_index=False
    )
    .agg(
        samples=(target_label, 'size'),
        conv_7d=(target_label, 'mean')
    )
    .sort_values(['conv_7d', 'samples'], ascending=[False, False])
)

display(combo_focus_click_sku.head(30))

,sku_focus_flag,click_cnt_bucket,distinct_sku_bucket,samples,conv_7d
37,1,4-5,6-10,24,0.458333
38,1,6-10,1-2,2738,0.398466
35,1,4-5,1-2,4180,0.397368
33,1,3,1-2,4373,0.393323
39,1,6-10,3-5,2441,0.390414
40,1,6-10,6-10,316,0.382911
36,1,4-5,3-5,2202,0.361490
42,1,10+,3-5,1030,0.356311
34,1,3,3-5,1241,0.350524
43,1,10+,6-10,514,0.350195


In [18]:
combo_focus_click_sku_filtered = combo_focus_click_sku[
    combo_focus_click_sku['samples'] >= 500
].copy()

display(combo_focus_click_sku_filtered.head(30))

,sku_focus_flag,click_cnt_bucket,distinct_sku_bucket,samples,conv_7d
38,1,6-10,1-2,2738,0.398466
35,1,4-5,1-2,4180,0.397368
33,1,3,1-2,4373,0.393323
39,1,6-10,3-5,2441,0.390414
36,1,4-5,3-5,2202,0.361490
42,1,10+,3-5,1030,0.356311
34,1,3,3-5,1241,0.350524
43,1,10+,6-10,514,0.350195
31,1,2,1-2,8007,0.337455
41,1,10+,1-2,743,0.306864


In [19]:
combo_focus_sku_channel = (
    analysis_df
    .groupby(
        ['sku_focus_flag', 'distinct_sku_bucket', 'distinct_channel_bucket'],
        observed=True,
        as_index=False
    )
    .agg(
        samples=(target_label, 'size'),
        conv_7d=(target_label, 'mean')
    )
    .sort_values(['conv_7d', 'samples'], ascending=[False, False])
)

combo_focus_sku_channel_filtered = combo_focus_sku_channel[
    combo_focus_sku_channel['samples'] >= 500
].copy()

display(combo_focus_sku_channel_filtered.head(30))

,sku_focus_flag,distinct_sku_bucket,distinct_channel_bucket,samples,conv_7d
19,1,1-2,2,893,0.513998
1,0,1-2,2,961,0.375650
24,1,6-10,1,835,0.366467
21,1,3-5,1,7758,0.354473
18,1,1-2,1,37894,0.304534
4,0,3-5,2,2695,0.197774
0,0,1-2,1,7483,0.120674
8,0,6-10,2,3519,0.117079
3,0,3-5,1,46454,0.115146
13,0,11-20,3,923,0.099675


In [20]:
combo_by_channel = (
    analysis_df
    .groupby(
        ['channel', 'sku_focus_flag', 'click_cnt_bucket', 'distinct_sku_bucket'],
        observed=True,
        as_index=False
    )
    .agg(
        samples=(target_label, 'size'),
        conv_7d=(target_label, 'mean')
    )
)

combo_by_channel_filtered = combo_by_channel[
    combo_by_channel['samples'] >= 300
].copy()

display(combo_by_channel_filtered.sort_values(['channel', 'conv_7d'], ascending=[True, False]).head(50))

,channel,sku_focus_flag,click_cnt_bucket,distinct_sku_bucket,samples,conv_7d
38,app,1,6-10,1-2,2478,0.399516
35,app,1,4-5,1-2,3650,0.392329
39,app,1,6-10,3-5,2208,0.390851
33,app,1,3,1-2,3738,0.377742
42,app,1,10+,3-5,983,0.356053
36,app,1,4-5,3-5,1899,0.352817
43,app,1,10+,6-10,486,0.345679
34,app,1,3,3-5,1053,0.326686
31,app,1,2,1-2,6746,0.320190
41,app,1,10+,1-2,691,0.311143


In [21]:
combo_by_type = (
    analysis_df
    .groupby(
        ['type_name', 'sku_focus_flag', 'click_cnt_bucket', 'distinct_sku_bucket'],
        observed=True,
        as_index=False
    )
    .agg(
        samples=(target_label, 'size'),
        conv_7d=(target_label, 'mean')
    )
)

combo_by_type_filtered = combo_by_type[
    combo_by_type['samples'] >= 300
].copy()

display(combo_by_type_filtered.sort_values(['type_name', 'conv_7d'], ascending=[True, False]).head(50))

,type_name,sku_focus_flag,click_cnt_bucket,distinct_sku_bucket,samples,conv_7d
35,1P,1,4-5,1-2,3007,0.405720
38,1P,1,6-10,1-2,1967,0.402644
33,1P,1,3,1-2,3083,0.402530
39,1P,1,6-10,3-5,1809,0.400221
36,1P,1,4-5,3-5,1589,0.372561
34,1P,1,3,3-5,878,0.372437
42,1P,1,10+,3-5,773,0.337646
31,1P,1,2,1-2,5593,0.335777
43,1P,1,10+,6-10,401,0.314214
32,1P,1,2,3-5,670,0.311940


In [22]:
top_combo_all = combo_focus_click_sku_filtered.sort_values(
    ['conv_7d', 'samples'],
    ascending=[False, False]
).copy()

display(top_combo_all.head(20))

,sku_focus_flag,click_cnt_bucket,distinct_sku_bucket,samples,conv_7d
38,1,6-10,1-2,2738,0.398466
35,1,4-5,1-2,4180,0.397368
33,1,3,1-2,4373,0.393323
39,1,6-10,3-5,2441,0.390414
36,1,4-5,3-5,2202,0.361490
42,1,10+,3-5,1030,0.356311
34,1,3,3-5,1241,0.350524
43,1,10+,6-10,514,0.350195
31,1,2,1-2,8007,0.337455
41,1,10+,1-2,743,0.306864


In [23]:
top_combo_by_channel = (
    combo_by_channel_filtered
    .sort_values(['channel', 'conv_7d', 'samples'], ascending=[True, False, False])
    .groupby('channel', as_index=False)
    .head(10)
)

display(top_combo_by_channel)

,channel,sku_focus_flag,click_cnt_bucket,distinct_sku_bucket,samples,conv_7d
38,app,1,6-10,1-2,2478,0.399516
35,app,1,4-5,1-2,3650,0.392329
39,app,1,6-10,3-5,2208,0.390851
33,app,1,3,1-2,3738,0.377742
42,app,1,10+,3-5,983,0.356053
36,app,1,4-5,3-5,1899,0.352817
43,app,1,10+,6-10,486,0.345679
34,app,1,3,3-5,1053,0.326686
31,app,1,2,1-2,6746,0.320190
41,app,1,10+,1-2,691,0.311143


In [24]:
top_combo_by_type = (
    combo_by_type_filtered
    .sort_values(['type_name', 'conv_7d', 'samples'], ascending=[True, False, False])
    .groupby('type_name', as_index=False)
    .head(10)
)

display(top_combo_by_type)

,type_name,sku_focus_flag,click_cnt_bucket,distinct_sku_bucket,samples,conv_7d
35,1P,1,4-5,1-2,3007,0.405720
38,1P,1,6-10,1-2,1967,0.402644
33,1P,1,3,1-2,3083,0.402530
39,1P,1,6-10,3-5,1809,0.400221
36,1P,1,4-5,3-5,1589,0.372561
34,1P,1,3,3-5,878,0.372437
42,1P,1,10+,3-5,773,0.337646
31,1P,1,2,1-2,5593,0.335777
43,1P,1,10+,6-10,401,0.314214
32,1P,1,2,3-5,670,0.311940
